In [1]:
import numpy as np

#########################     生成假数据     ###############################
ell_ = np.arange(64)
C_TT = 100 * (1.0 + ell_)**(-1.0)  # Convert to float
C_EE = 50 * (1.0 + ell_)**(-1.2)
C_TE = 30 * (1.0 + ell_)**(-1.1)
C_GG = 200 * (1.0 + ell_)**(-0.8)
C_TG = 40 * (1.0 + ell_)**(-0.9)
C_EG = 20 * (1.0 + ell_)**(-1.0)
Fake_N_GG = 0.1 * np.ones(64)


partial_C_TT_partial_fnl = 1.0 * (1.0 + ell_)**(-1.0)
partial_C_EE_partial_fnl = 1.0 * (1.0 + ell_)**(-1.2)
partial_C_TE_partial_fnl = 1.0 * (1.0 + ell_)**(-1.1)   
partial_C_GG_partial_fnl = 1.0 * (1.0 + ell_)**(-0.8)
partial_C_TG_partial_fnl = 1.0 * (1.0 + ell_)**(-0.9)
partial_C_EG_partial_fnl = 1.0 * (1.0 + ell_)**(-1.0)
######################     假数据部分结束     ############################


def compute_covariance_matrices(ell,N_GG=Fake_N_GG):
    """
    计算单个ell值的协方差矩阵
    
    参数:
    ell: int, 单个ell值
    
    返回:
    C_ell: ndarray, 形状为(6, 6)的协方差矩阵
    
    注意：假设 C_TT, C_EE, C_TE, C_GG, C_TG, C_EG, N_GG 已经定义为全局变量
    """
    # 初始化6x6矩阵
    C_ell = np.zeros((6, 6))
    
    # 计算公共因子
    factor = 1.0 / (2 * ell + 1)
    factor2 = 2 * factor
    
    # 按照矩阵位置排序：[TT, EE, TE, GG, TG, EG]
    
    # 前三行三列保持为0
    
    # TT-GG, EE-GG, TE-GG (第4列)
    C_ell[0, 3] = factor2 * C_TG[ell]**2
    C_ell[1, 3] = factor2 * C_EG[ell]**2
    C_ell[2, 3] = factor2 * C_TG[ell] * C_EG[ell]
    
    # GG-GG (对角线)
    C_ell[3, 3] = factor2 * (C_GG[ell] + N_GG[ell])**2
    
    # TT-TG, EE-TG, TE-TG, GG-TG (第5列)
    C_ell[0, 4] = factor2 * C_TT[ell] * C_TG[ell]
    C_ell[1, 4] = factor2 * C_TE[ell] * C_EG[ell]
    C_ell[2, 4] = factor * (C_TT[ell] * C_EG[ell] + C_TE[ell] * C_TG[ell])
    C_ell[3, 4] = factor2 * (C_GG[ell] + N_GG[ell]) * C_TG[ell]
    
    # TG-TG (对角线)
    C_ell[4, 4] = factor * (C_TG[ell]**2 + C_TT[ell] * (C_GG[ell] + N_GG[ell]))
    
    # TT-EG, EE-EG, TE-EG, GG-EG, TG-EG (第6列)
    C_ell[0, 5] = factor2 * C_TE[ell] * C_TG[ell]
    C_ell[1, 5] = factor2 * C_EE[ell] * C_EG[ell]
    C_ell[2, 5] = factor * (C_TE[ell] * C_EG[ell] + C_TG[ell] * C_EE[ell])
    C_ell[3, 5] = factor2 * (C_GG[ell] + N_GG[ell]) * C_EG[ell]
    C_ell[4, 5] = factor * (C_TE[ell] * (C_GG[ell] + N_GG[ell]) + C_EG[ell] * C_TG[ell])
    
    # EG-EG (对角线)
    C_ell[5, 5] = factor * (C_EG[ell]**2 + C_EE[ell] * (C_GG[ell] + N_GG[ell]))
    
    # 利用对称性填充下三角部分
    for i in range(6):
        for j in range(i):
            C_ell[i, j] = C_ell[j, i]
    
    return C_ell


def compute_fisher_matrix(N_GG=Fake_N_GG):
    """
    计算Fisher矩阵（在这种情况下是一个标量）
    
    参数:
    ell_min: int, 最小的ell值
    ell_max: int, 最大的ell值
    
    返回:
    F: float, Fisher矩阵元素
    
    注意：假设partial_C_XX_partial_fnl已经定义为全局变量
    """
    ell_min = 2
    ell_max = 2 + len(C_TT) - 1
    print("ell_min: ", ell_min)
    print("ell_max: ", ell_max)
    F = 0.0
    
    for ell in range(ell_min, ell_max + 1):
        # 获取ell对应的协方差矩阵
        C_ell = compute_covariance_matrices(ell)
        
        # 计算协方差矩阵的逆
        C_ell_inv = np.linalg.inv(C_ell)
        
        # 构建导数向量 (6,)
        dC = np.array([0.0, 0.0, 0.0, 
                      partial_C_GG_partial_fnl[ell],
                      partial_C_TG_partial_fnl[ell],
                      partial_C_EG_partial_fnl[ell]])
        
        # 将dC重塑为行向量 (1,6) 和列向量 (6,1)
        dC_row = dC.reshape(1, -1)
        dC_col = dC.reshape(-1, 1)
        print('行矩阵：', dC_row)
        print('列矩阵：', dC_col)
        
        # 计算 (1,6) @ (6,6) @ (6,1) 得到标量
        F += dC_row @ C_ell_inv @ dC_col

        
    return F